<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Nano Action: Policy with vLLM-Omni

## Prerequisites

Generator requires the Guardrail. Request access to the gated [nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) HF repository before running guarded examples. This notebook disables guardrails for the policy requests with `guardrails: false` / `--no-guardrails`.

## Overview

This notebook runs Cosmos3 Nano **action policy** inference through vLLM-Omni using the checked-in DROID LeRobot sample under `assets/droid_lerobot_example`.

It covers two policy serving paths from the vLLM-Omni Cosmos3 recipe:

- `POST /v1/videos`: first frame plus instruction -> rollout video plus top-level `action` metadata.
- `ws://.../v1/realtime/robot/openpi`: OpenPI-compatible observation -> action chunk, suitable for a simulated or real robot client.

The OpenPI example sends one static sample observation. Real robot clients should replace the zero joint/gripper state with live state and send repeated observations in a control loop.

## Start vLLM-Omni Policy Server

Start the server in a terminal from the `cosmos` repo root. The container listens on port `8000`; Docker publishes it to host port `8001`, so the notebook uses `http://localhost:8001`.

The OpenPI RoboLab branch reuses reference Cosmos Framework action transforms. The command below assumes the framework checkout is mounted at `/workspace/cosmos-framework`; adjust `PYTHONPATH` if your checkout lives elsewhere.

```bash
cat > /tmp/cosmos3_droid_openpi_stage_overrides.json <<'JSON'
{
  "0": {
    "model_config": {
      "policy_server_config": {
        "image_resolution": [540, 640],
        "n_external_cameras": 2,
        "needs_wrist_camera": true,
        "needs_stereo_camera": false,
        "needs_session_id": true,
        "action_space": "joint_position"
      }
    }
  }
}
JSON

docker rm -f cosmos3-vllm-omni-policy-notebook 2>/dev/null || true

docker run -d --name cosmos3-vllm-omni-policy-notebook   --runtime nvidia --gpus '"device=0"'   -e CUDA_DEVICE_ORDER=PCI_BUS_ID   -e PYTHONPATH=/workspace/cosmos-framework   -v "/mnt/sdb/.cache/huggingface:/root/.cache/huggingface"   -v "$PWD:/workspace"   -p 8001:8000 --ipc=host   vllm/vllm-omni:cosmos3   vllm serve nvidia/Cosmos3-Nano-Policy-DROID     --omni     --model-class-name Cosmos3OmniDiffusersPipeline     --allowed-local-media-path /     --port 8000     --init-timeout 1800     --no-guardrails     --stage-overrides "$(cat /tmp/cosmos3_droid_openpi_stage_overrides.json)"

# Wait until this returns model metadata before running the inference cells.
curl http://localhost:8001/v1/models
```

To inspect startup logs:

```bash
docker logs -f cosmos3-vllm-omni-policy-notebook
```

In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def http_to_ws_url(base_url: str) -> str:
    if base_url.startswith("https://"):
        ws_base = "wss://" + base_url[len("https://"):]
    elif base_url.startswith("http://"):
        ws_base = "ws://" + base_url[len("http://"):]
    else:
        ws_base = base_url
    return ws_base.rstrip("/") + "/v1/realtime/robot/openpi"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_VLLM_OUTPUT_ROOT", COSMOS_ROOT / "outputs" / "cosmos3_action_vllm")
).resolve()
COSMOS3_INPUT_DIR = COSMOS3_OUTPUT_ROOT / "inputs"
COSMOS3_POLICY_OUTPUT_DIR = COSMOS3_OUTPUT_ROOT / "action_policy_droid"
COSMOS3_ACTION_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "action"
DROID_ASSET_ROOT = COSMOS3_ACTION_ROOT / "assets" / "droid_lerobot_example"
VLLM_BASE_URL = os.environ.get("COSMOS3_VLLM_BASE_URL", "http://localhost:8001").rstrip("/")
OPENPI_WS_URL = os.environ.get("COSMOS3_OPENPI_WS_URL", http_to_ws_url(VLLM_BASE_URL))
VLLM_MODEL = os.environ.get("COSMOS3_VLLM_MODEL", "nvidia/Cosmos3-Nano-Policy-DROID")

COSMOS3_INPUT_DIR.mkdir(parents=True, exist_ok=True)
COSMOS3_POLICY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("COSMOS_ROOT:", COSMOS_ROOT)
print("DROID_ASSET_ROOT:", DROID_ASSET_ROOT)
print("COSMOS3_INPUT_DIR:", COSMOS3_INPUT_DIR)
print("COSMOS3_POLICY_OUTPUT_DIR:", COSMOS3_POLICY_OUTPUT_DIR)
print("COSMOS3_VLLM_BASE_URL:", VLLM_BASE_URL)
print("COSMOS3_OPENPI_WS_URL:", OPENPI_WS_URL)
print("COSMOS3_VLLM_MODEL:", VLLM_MODEL)

## Prepare a DROID Policy Input

This cell extracts the first frame from each checked-in DROID camera video, creates a 640x540 multiview conditioning image for the video API, and builds a matching OpenPI observation dictionary.

The OpenPI observation uses real camera frames from the sample asset and zero joint/gripper state so the cell can run without a LeRobot/parquet dependency in the notebook kernel.

In [ ]:
import subprocess

import numpy as np
from PIL import Image, ImageOps
from IPython.display import display

try:
    import imageio_ffmpeg
except ImportError as exc:
    raise RuntimeError("Install imageio-ffmpeg in this notebook kernel: pip install imageio-ffmpeg") from exc

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
CAMERA_VIDEO_PATHS = {
    "observation/wrist_image_left": DROID_ASSET_ROOT / "videos" / "observation.image.wrist_image_left" / "chunk-000" / "file-000.mp4",
    "observation/exterior_image_1_left": DROID_ASSET_ROOT / "videos" / "observation.image.exterior_image_1_left" / "chunk-000" / "file-000.mp4",
    "observation/exterior_image_2_left": DROID_ASSET_ROOT / "videos" / "observation.image.exterior_image_2_left" / "chunk-000" / "file-000.mp4",
}
for key, video_path in CAMERA_VIDEO_PATHS.items():
    assert video_path.exists(), f"missing {key}: {video_path}"


def extract_first_frame(video_path: Path, out_path: Path) -> Path:
    if not out_path.exists() or out_path.stat().st_mtime < video_path.stat().st_mtime:
        subprocess.run(
            [FFMPEG, "-y", "-loglevel", "error", "-i", str(video_path), "-frames:v", "1", str(out_path)],
            check=True,
        )
    return out_path


frame_paths = {
    key: extract_first_frame(video_path, COSMOS3_INPUT_DIR / f"policy_{key.split('/')[-1]}.png")
    for key, video_path in CAMERA_VIDEO_PATHS.items()
}
frames = {key: Image.open(path).convert("RGB") for key, path in frame_paths.items()}

# DROID policy uses a concatenated multi-view frame. Keep the OpenPI branch on
# raw per-camera frames; use this composed image only for the /v1/videos path.
target_w, target_h = 640, 540
top_h = target_h // 2
bottom_h = target_h - top_h
half_w = target_w // 2
wrist = ImageOps.fit(frames["observation/wrist_image_left"], (target_w, top_h), method=Image.Resampling.BICUBIC)
left = ImageOps.fit(frames["observation/exterior_image_1_left"], (half_w, bottom_h), method=Image.Resampling.BICUBIC)
right = ImageOps.fit(frames["observation/exterior_image_2_left"], (half_w, bottom_h), method=Image.Resampling.BICUBIC)
policy_image = Image.new("RGB", (target_w, target_h))
policy_image.paste(wrist, (0, 0))
policy_image.paste(left, (0, top_h))
policy_image.paste(right, (half_w, top_h))
policy_image_path = COSMOS3_INPUT_DIR / "droid_policy_first_frame.png"
policy_image.save(policy_image_path)

policy_prompt = os.environ.get(
    "COSMOS3_POLICY_PROMPT",
    "Pick up the object and place it in the target container.",
)
openpi_observation = {
    "prompt": policy_prompt,
    "session_id": os.environ.get("COSMOS3_POLICY_SESSION_ID", "cosmos3-policy-notebook"),
    "observation/wrist_image_left": np.asarray(frames["observation/wrist_image_left"], dtype=np.uint8),
    "observation/exterior_image_1_left": np.asarray(frames["observation/exterior_image_1_left"], dtype=np.uint8),
    "observation/exterior_image_2_left": np.asarray(frames["observation/exterior_image_2_left"], dtype=np.uint8),
    "observation/joint_position": np.zeros(7, dtype=np.float32),
    "observation/gripper_position": np.zeros(1, dtype=np.float32),
}

print("policy prompt:", policy_prompt)
print("video API conditioning image:", policy_image_path, policy_image.size)
for key, value in openpi_observation.items():
    if isinstance(value, np.ndarray):
        print(key, value.shape, value.dtype)

display(policy_image)

## Run Policy Inference Through `/v1/videos`

This path behaves like the forward/inverse vLLM action notebooks: it sends a multipart request to the OpenAI-compatible video API, polls the async job, writes the generated rollout video, and saves the predicted action from the response metadata.

In [ ]:
import json
import time
from pathlib import Path

try:
    import requests
except ImportError as exc:
    raise RuntimeError("Install requests in this notebook kernel: pip install requests") from exc


def check_vllm_server(timeout_s: int = 600, interval_s: int = 10) -> None:
    deadline = time.time() + timeout_s
    last_error: Exception | None = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{VLLM_BASE_URL}/v1/models", timeout=10)
            response.raise_for_status()
            print(response.json())
            return
        except requests.RequestException as exc:
            last_error = exc
            print(f"Waiting for vLLM server at {VLLM_BASE_URL}: {exc}")
            time.sleep(interval_s)
    raise RuntimeError(
        f"vLLM server did not become ready at {VLLM_BASE_URL} within {timeout_s}s. "
        "Check `docker logs -f cosmos3-vllm-omni-policy-notebook`."
    ) from last_error


ACTION_VIDEO_RES_SIZE_INFO = {
    "480": {
        "1,1": (640, 640),
        "4,3": (736, 544),
        "3,4": (544, 736),
        "16,9": (832, 480),
        "9,16": (480, 832),
    }
}


def closest_action_size(height: int, width: int, resolution: str = "480") -> tuple[int, int]:
    input_ratio = height / width
    candidates = ACTION_VIDEO_RES_SIZE_INFO[resolution].values()
    return min(candidates, key=lambda size: abs(input_ratio - size[1] / size[0]))


def submit_policy_video() -> dict:
    run_dir = COSMOS3_POLICY_OUTPUT_DIR / "video_api"
    run_dir.mkdir(parents=True, exist_ok=True)

    input_width, input_height = Image.open(policy_image_path).size
    target_width, target_height = closest_action_size(input_height, input_width)
    extra_params = {
        "action_mode": "policy",
        "domain_name": "droid_lerobot",
        "raw_action_dim": 10,
        "action_chunk_size": 16,
        "image_size": 480,
        "view_point": "concat_view",
        "guardrails": False,
    }
    form = {
        "model": VLLM_MODEL,
        "prompt": policy_prompt,
        "num_frames": 17,
        "fps": 15,
        "size": f"{target_width}x{target_height}",
        "num_inference_steps": 30,
        "guidance_scale": 1.0,
        "flow_shift": 5.0,
        "seed": 0,
        "extra_params": json.dumps(extra_params),
    }

    with policy_image_path.open("rb") as image_file:
        response = requests.post(
            f"{VLLM_BASE_URL}/v1/videos",
            data={key: str(value) for key, value in form.items()},
            files={"input_reference": (policy_image_path.name, image_file, "image/png")},
            timeout=120,
        )
    if not response.ok:
        (run_dir / "error_response.txt").write_text(response.text)
        print("vLLM request failed:", response.status_code)
        print(response.text)
        print("form:", json.dumps(form, indent=2))
        response.raise_for_status()

    initial = response.json()
    (run_dir / "response.json").write_text(json.dumps(initial, indent=2))

    while True:
        response = requests.get(f"{VLLM_BASE_URL}/v1/videos/{initial['id']}", timeout=30)
        response.raise_for_status()
        final = response.json()
        (run_dir / "final.json").write_text(json.dumps(final, indent=2))
        print(initial["id"], final.get("status"), f"{final.get('progress', 0)}%")
        if final.get("status") == "completed":
            break
        if final.get("status") in {"failed", "cancelled"}:
            raise RuntimeError(json.dumps(final, indent=2))
        time.sleep(2)

    action = final.get("action")
    if not action or "data" not in action:
        raise RuntimeError(f"vLLM response did not include action data: {json.dumps(final, indent=2)}")
    (run_dir / "action.json").write_text(json.dumps(action, indent=2))
    sample_outputs = {"outputs": [{"content": {"action": action["data"]}}]}
    (run_dir / "sample_outputs.json").write_text(json.dumps(sample_outputs, indent=2))

    content_response = requests.get(f"{VLLM_BASE_URL}/v1/videos/{initial['id']}/content", timeout=300)
    content_response.raise_for_status()
    video_path = run_dir / "policy_rollout.mp4"
    if content_response.content:
        video_path.write_bytes(content_response.content)
        print("saved", video_path)
    else:
        video_path = None
        print("video content endpoint returned an empty body")

    print("saved", run_dir / "action.json")
    print("action shape:", action.get("shape"), "dtype:", action.get("dtype"), "domain_id:", action.get("domain_id"))
    return {"initial": initial, "final": final, "run_dir": run_dir, "video_path": video_path, "action": action}


check_vllm_server()
policy_video_result = submit_policy_video()

## Inspect Video API Outputs

Preview the rollout video if the server returned one, and print the first few predicted action rows.

In [ ]:
import subprocess

import imageio_ffmpeg
from IPython.display import Video, display

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()


def make_preview(src: Path, crf: int = 28) -> Path:
    preview = src.with_name(f"{src.stem}_preview.mp4")
    if not preview.exists() or preview.stat().st_mtime < src.stat().st_mtime:
        subprocess.run(
            [
                FFMPEG,
                "-y",
                "-loglevel",
                "error",
                "-i",
                str(src),
                "-c:v",
                "libx264",
                "-crf",
                str(crf),
                "-preset",
                "veryfast",
                "-an",
                "-pix_fmt",
                "yuv420p",
                str(preview),
            ],
            check=True,
        )
    return preview


action = policy_video_result["action"]
action_array = np.asarray(action["data"], dtype=np.float32)
print("action array:", action_array.shape, action_array.dtype)
print(action_array[: min(5, len(action_array))])

video_path = policy_video_result.get("video_path")
if video_path is not None:
    preview = make_preview(video_path)
    print(f"preview: {preview}")
    display(Video(str(preview), embed=True))

## Run Policy Inference Through OpenPI WebSocket

The OpenPI endpoint is for robot control clients. It sends model-specific metadata immediately after connection, then accepts msgpack-numpy observation dictionaries and returns action chunks.

Install the small client dependencies in this notebook kernel if needed:

```bash
pip install websockets openpi-client
```

In [ ]:
try:
    import websockets.sync.client as websockets_client
    from openpi_client import msgpack_numpy
except ImportError as exc:
    raise RuntimeError("Install OpenPI client deps in this notebook kernel: pip install websockets openpi-client") from exc


def to_jsonable(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def summarize_actions(actions) -> None:
    if isinstance(actions, dict):
        for key, value in actions.items():
            arr = np.asarray(value, dtype=np.float32)
            print(key, arr.shape, arr.dtype)
            print(arr.reshape(-1, arr.shape[-1])[: min(3, arr.reshape(-1, arr.shape[-1]).shape[0])])
    else:
        arr = np.asarray(actions, dtype=np.float32)
        print("actions", arr.shape, arr.dtype)
        if arr.ndim >= 2:
            print(arr.reshape(-1, arr.shape[-1])[: min(3, arr.reshape(-1, arr.shape[-1]).shape[0])])
        else:
            print(arr[: min(10, arr.shape[0])])


def run_openpi_policy(obs: dict) -> dict:
    packer = msgpack_numpy.Packer()
    ws = websockets_client.connect(
        OPENPI_WS_URL,
        compression=None,
        max_size=None,
        ping_interval=300,
        ping_timeout=3600,
    )
    try:
        metadata_payload = ws.recv()
        if isinstance(metadata_payload, str):
            raise RuntimeError(f"Expected binary metadata payload, got text: {metadata_payload}")
        metadata = msgpack_numpy.unpackb(metadata_payload)
        if not isinstance(metadata, dict):
            raise RuntimeError(f"Expected metadata dict, got {type(metadata)!r}")
        print("server metadata:", metadata)

        payload = dict(obs)
        payload["endpoint"] = "infer"
        ws.send(packer.pack(payload))
        response_payload = ws.recv()
        if isinstance(response_payload, str):
            raise RuntimeError(f"Inference failed: {response_payload}")
        actions = msgpack_numpy.unpackb(response_payload)
        if isinstance(actions, dict) and actions.get("type") == "error":
            raise RuntimeError(f"Inference failed: {actions.get('message', actions)}")

        ws.send(packer.pack({"endpoint": "reset"}))
        reset_payload = ws.recv()
        reset_response = msgpack_numpy.unpackb(reset_payload) if not isinstance(reset_payload, str) else reset_payload
        print("reset response:", reset_response)
        return {"metadata": metadata, "actions": actions, "reset_response": reset_response}
    finally:
        ws.close()


openpi_result = run_openpi_policy(openpi_observation)
summarize_actions(openpi_result["actions"])

openpi_output_dir = COSMOS3_POLICY_OUTPUT_DIR / "openpi"
openpi_output_dir.mkdir(parents=True, exist_ok=True)
(openpi_output_dir / "result.json").write_text(json.dumps(to_jsonable(openpi_result), indent=2))
print("saved", openpi_output_dir / "result.json")

## Use with RoboLab

For an interactive simulation, keep the same vLLM-Omni server running and point a RoboLab Cosmos3 policy client at `ws://localhost:8001/v1/realtime/robot/openpi`.

```bash
python policies/cosmos3/run.py --task BananaInBowlTask
```

Use `python policies/cosmos3/run.py --help` in the RoboLab repo to check how your checkout configures the policy endpoint.